<!-- RAG with PDF data extraction  -->

In [ ]:
!pip install pypdf

In [ ]:
import os

from dotenv import load_dotenv
load = load_dotenv(".env")

In [ ]:
from langchain_ollama import ChatOllama

llm = ChatOllama(
    base_url="http://localhost:11434",
    model="qwen3:8b",
    temperature=0.5,
    num_predict=2202
)


In [ ]:
# Ectracting PDF files
from langchain_community.document_loaders import PyPDFLoader

pdf1 = "attention.pdf"
pdf2 = "LLMForgetting.pdf"
pdf3 = "TestingAndEvaluatingLLM.pdf"
pdf4 = "user_Profile.pdf.pdf"

pdfFiles = [pdf1, pdf2, pdf3, pdf4]

documents = []

for pdf in pdfFiles:
    loader = PyPDFLoader(pdf)
    documents.extend(loader.load())

print(f"Total number of pages in all PDFs: {len(documents)}")

In [ ]:
# Text Splitting

from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200, add_start_index=True)

all_split_docs = text_splitter.split_documents(documents)

len(all_split_docs)  # Total number of chunks after splitting the documents


In [ ]:
# Embedding the chunks

from langchain_core.embeddings import Embeddings
from langchain_ollama import OllamaEmbeddings

# Ollama can fail when a large document list is sent in one request.
ollama_embeddings = OllamaEmbeddings(model="nomic-embed-text")


class BatchedOllamaEmbeddings(Embeddings):
    def __init__(self, embedding_model, batch_size=8):
        self.embedding_model = embedding_model
        self.batch_size = batch_size

    def embed_documents(self, texts):
        vectors = []
        for start in range(0, len(texts), self.batch_size):
            batch = texts[start:start + self.batch_size]
            vectors.extend(self.embedding_model.embed_documents(batch))
        return vectors

    def embed_query(self, text):
        return self.embedding_model.embed_query(text)


embeddings = BatchedOllamaEmbeddings(ollama_embeddings)

vector_one = embeddings.embed_query(all_split_docs[0].page_content)
vector_two = embeddings.embed_query(all_split_docs[1].page_content)

print(len(vector_one))
print(len(vector_two))

In [ ]:
# Vector store

from langchain_chroma import Chroma

vector_store = Chroma.from_documents(
    documents=all_split_docs,
    embedding=embeddings,
    persist_directory="./chroma_langchain_db_v3",
)

In [ ]:
# Retrieve relevant chunks

from langchain_chroma import Chroma

vector_store = Chroma(
    persist_directory="./chroma_langchain_db_v3",
    embedding_function=embeddings,
)

question = "What is my overall AI career direction?"
retrieved_docs = vector_store.similarity_search(question, k=3)

retrieved_docs

In [ ]:
# Generate an answer from the retrieved context

context = "\n\n".join(doc.page_content for doc in retrieved_docs)

prompt = f"""Answer the question using only the context below.
If the answer is not present in the context, say: I don't know based on the documents.

Context:
{context}

Question: {question}
Answer:"""

response = llm.invoke(prompt)
print(response.content)

In [ ]:
# Create a profile-specific retriever

retriever = vector_store.as_retriever(
    search_type="similarity",
    search_kwargs={
        "k": 3,
        "filter": {"source": "user_Profile.pdf"},
    },
)

retriever.invoke("What programming languages are listed in my profile?")

In [ ]:
# Full RetrievalQA implementation

from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate

question = "What programming languages are listed in my profile?"

prompt = ChatPromptTemplate.from_template("""Use only the context below to answer the question.
If the answer is not in the context, say: I don't know based on the documents.
Keep the answer concise and do not add unsupported details.

Context:
{context}

Question: {question}
Answer:""")


def format_documents(documents):
    return "\n\n".join(document.page_content for document in documents)


retrieval_qa_chain = (
    {
        "context": retriever | format_documents,
        "question": lambda value: value,
    }
    | prompt
    | llm
    | StrOutputParser()
)

answer = retrieval_qa_chain.invoke(question)
print(answer)

In [ ]:
# RetrivalQA

